In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline
# Lazimli kitabxanalari import edirem

In [2]:
import os
files  = os.listdir()
print(files)
# Burda directorydaki fayllara baxiram 

['.git', '.ipynb_checkpoints', 'AEP_hourly.csv', 'COMED_hourly.csv', 'DAYTON_hourly.csv', 'DEOK_hourly.csv', 'DOM_hourly.csv', 'DUQ_hourly.csv', 'EKPC_hourly.csv', 'Energy_Consumption.ipynb', 'est_hourly.paruqet', 'FE_hourly.csv', 'NI_hourly.csv', 'PJME_hourly.csv', 'PJMW_hourly.csv', 'pjm_hourly_est.csv', 'PJM_Load_hourly.csv', 'Risk Prediction for Patient Readmission.ipynb']


In [3]:
all_dfs = []

for file in os.listdir():
    if file.endswith(".csv") and file != 'pjm_hourly_est.csv':
        df = pd.read_csv(os.path.join(file))
        region = df.columns[1].replace('_MW','')
        df.columns = ["Datetime", "Load"]
        df["regions"] = region
        all_dfs.append(df)
df = pd.concat(all_dfs,ignore_index = True)
df['Datetime'] = pd.to_datetime(df['Datetime'],utc=True)
# Burda pjm_hourly_est.csv den basqa mene lazim olan csv filleri gotururem ve birlesdirirem 

In [4]:
df.head()

,Datetime,Load,regions
0,2004-12-31 01:00:00+00:00,13478.0,AEP
1,2004-12-31 02:00:00+00:00,12865.0,AEP
2,2004-12-31 03:00:00+00:00,12577.0,AEP
3,2004-12-31 04:00:00+00:00,12517.0,AEP
4,2004-12-31 05:00:00+00:00,12670.0,AEP


In [5]:
df.columns

Index(['Datetime', 'Load', 'regions'], dtype='object')

In [6]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1090167 entries, 0 to 1090166
Data columns (total 3 columns):
 #   Column    Non-Null Count    Dtype              
---  ------    --------------    -----              
 0   Datetime  1090167 non-null  datetime64[ns, UTC]
 1   Load      1090167 non-null  float64            
 2   regions   1090167 non-null  object             
dtypes: datetime64[ns, UTC](1), float64(1), object(1)
memory usage: 25.0+ MB


In [7]:
df['Hour'] = df['Datetime'].dt.hour
df['Day'] = df['Datetime'].dt.day
df['Months'] = df['Datetime'].dt.month
df['Year'] = df['Datetime'].dt.year
df['Day_of_week'] = df['Datetime'].dt.dayofweek
# burda biz datetime ayirmaliyiq cunki her time-based features dataya tesir edir meselen aylara boldukde fesillere gore enerji istifadesi

In [8]:
df.head()

,Datetime,Load,regions,Hour,Day,Months,Year,Day_of_week
0,2004-12-31 01:00:00+00:00,13478.0,AEP,1,31,12,2004,4
1,2004-12-31 02:00:00+00:00,12865.0,AEP,2,31,12,2004,4
2,2004-12-31 03:00:00+00:00,12577.0,AEP,3,31,12,2004,4
3,2004-12-31 04:00:00+00:00,12517.0,AEP,4,31,12,2004,4
4,2004-12-31 05:00:00+00:00,12670.0,AEP,5,31,12,2004,4


In [9]:
df['lag1'] = df['Load'].shift(1)# kecmisdeki deyeri indiki zaman ucun feature kimi istifade edirik
df['lag24'] = df['Load'].shift(24)
df['rolling_mean'] = df['Load'].rolling(24).mean()
# Bunlar time series datasinda en vacib hisslereden biridir hansi ki dataya tesir edir

In [10]:
df['regions'].unique()

array(['AEP', 'COMED', 'DAYTON', 'DEOK', 'DOM', 'DUQ', 'EKPC', 'FE', 'NI',
       'PJME', 'PJMW', 'PJM_Load'], dtype=object)

In [11]:
df['regions'].isnull().sum()

np.int64(0)

In [12]:
df['regions'].nunique()

12

In [13]:
df = pd.get_dummies(df, columns=['regions'],dtype= int)
# burda regionlari her birini ayri sutunu yigiram ve True ucun 1 false ucun 0 deyeri verirem 

In [14]:
df.head()

,Datetime,Load,Hour,Day,Months,Year,Day_of_week,lag1,lag24,rolling_mean,...,regions_DAYTON,regions_DEOK,regions_DOM,regions_DUQ,regions_EKPC,regions_FE,regions_NI,regions_PJME,regions_PJMW,regions_PJM_Load
0,2004-12-31 01:00:00+00:00,13478.0,1,31,12,2004,4,NaN,NaN,NaN,...,0,0,0,0,0,0,0,0,0,0
1,2004-12-31 02:00:00+00:00,12865.0,2,31,12,2004,4,13478.0,NaN,NaN,...,0,0,0,0,0,0,0,0,0,0
2,2004-12-31 03:00:00+00:00,12577.0,3,31,12,2004,4,12865.0,NaN,NaN,...,0,0,0,0,0,0,0,0,0,0
3,2004-12-31 04:00:00+00:00,12517.0,4,31,12,2004,4,12577.0,NaN,NaN,...,0,0,0,0,0,0,0,0,0,0
4,2004-12-31 05:00:00+00:00,12670.0,5,31,12,2004,4,12517.0,NaN,NaN,...,0,0,0,0,0,0,0,0,0,0


In [15]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1090167 entries, 0 to 1090166
Data columns (total 22 columns):
 #   Column            Non-Null Count    Dtype              
---  ------            --------------    -----              
 0   Datetime          1090167 non-null  datetime64[ns, UTC]
 1   Load              1090167 non-null  float64            
 2   Hour              1090167 non-null  int32              
 3   Day               1090167 non-null  int32              
 4   Months            1090167 non-null  int32              
 5   Year              1090167 non-null  int32              
 6   Day_of_week       1090167 non-null  int32              
 7   lag1              1090166 non-null  float64            
 8   lag24             1090143 non-null  float64            
 9   rolling_mean      1090144 non-null  float64            
 10  regions_AEP       1090167 non-null  int64              
 11  regions_COMED     1090167 non-null  int64              
 12  regions_DAYTON    1090167 no

In [16]:
df.dropna(subset=['lag1', 'lag24', 'rolling_mean'], inplace=True)

In [17]:
df['lag1'].unique()

array([12892., 14097., 13667., ..., 46438., 46937., 20754.],
      shape=(48278,))

In [18]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 1090143 entries, 24 to 1090166
Data columns (total 22 columns):
 #   Column            Non-Null Count    Dtype              
---  ------            --------------    -----              
 0   Datetime          1090143 non-null  datetime64[ns, UTC]
 1   Load              1090143 non-null  float64            
 2   Hour              1090143 non-null  int32              
 3   Day               1090143 non-null  int32              
 4   Months            1090143 non-null  int32              
 5   Year              1090143 non-null  int32              
 6   Day_of_week       1090143 non-null  int32              
 7   lag1              1090143 non-null  float64            
 8   lag24             1090143 non-null  float64            
 9   rolling_mean      1090143 non-null  float64            
 10  regions_AEP       1090143 non-null  int64              
 11  regions_COMED     1090143 non-null  int64              
 12  regions_DAYTON    1090143 non-nu

In [19]:
from sklearn.model_selection import train_test_split
from xgboost import XGBRegressor
X = df.drop('Load',axis = 1)
y = df['Load']
train_size = int(len(df) * 0.8)
train = df.iloc[:train_size]
test = df.iloc[train_size:]
features = ['Hour', 'lag1', 'lag24','Day','Months','Year','Day_of_week','rolling_mean','regions_AEP','regions_COMED','regions_DAYTON','regions_DEOK'
           ,'regions_DOM','regions_DUQ','regions_EKPC','regions_FE','regions_NI','regions_PJME','regions_PJMW','regions_PJM_Load'
           ]

X_train = train[features]
y_train = train['Load']


X_test = test[features]
y_test = test['Load']
# burda time series datasi oldugu ucun train test split istifade etmek modeli sehv proqnozlasdirmaga getirib cixardigi ucun datani train ve teste ilco ile boluruk

In [20]:
xgb = XGBRegressor()

In [21]:
xgb.fit(X_train,y_train)

,objective,'reg:squarederror'
,base_score,None
,booster,None
,callbacks,None
,colsample_bylevel,None
,colsample_bynode,None
,colsample_bytree,None
,device,None
,early_stopping_rounds,None
,enable_categorical,False
,eval_metric,None


In [22]:
from sklearn.metrics import r2_score, mean_absolute_error,mean_squared_error,mean_absolute_percentage_error
#bizden teleb olunan metricleri import edirik

In [23]:
y_pred = xgb.predict(X_test)
print('R2 Score: ', r2_score(y_test,y_pred))
print('MAPE: ', mean_absolute_percentage_error(y_test,y_pred))
print('MAE: ', mean_absolute_error(y_test,y_pred))
print('MSE: ', mean_squared_error(y_test,y_pred))
#datani xgboost ile proqnozlasdiririq

R2 Score:  0.9896037384107309
MAPE:  0.08814475485815979
MAE:  618.8461033716684
MSE:  1618496.5503260226


In [24]:
from sklearn.ensemble import RandomForestRegressor

In [25]:
rfr = RandomForestRegressor()

In [26]:
rfr.fit(X_train,y_train)

,n_estimators,100
,criterion,'squared_error'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,1.0
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [27]:
y_pred2 = rfr.predict(X_test)
print('R2 Score: ', r2_score(y_test,y_pred2))
print('MAPE: ', mean_absolute_percentage_error(y_test,y_pred2))
print('MAE: ', mean_absolute_error(y_test,y_pred2))
print('MSE: ', mean_squared_error(y_test,y_pred2))

R2 Score:  0.9982429053093468
MAPE:  0.04096368989907503
MAE:  320.2032776373785
MSE:  273545.6078128964


In [ ]:
import shap
explainer = shap.TreeExplainer(rfr)
X_sample = X_test.sample(5000, random_state=42)
shap_values = explainer.shap_values(X_sample)
shap.summary_plot(shap_values, X_sample, plot_type="bar")

In [ ]:
#Datetime → zaman faktoru olduğu üçün yük istehlakında günlük və saatlıq patternləri tutmağa kömək edir
#Weather condition (sunny, cloudy, rain) → ümumi hava vəziyyətini göstərir və enerji istehlak patternlərinə təsir edir
#Hour → günün hansı saatı olduğunu göstərir və istehlakın pik və aşağı saatlarını müəyyən edir
#Day of week → həftə içi və həftə sonu fərqlərini tutmağa kömək edir
#Lag features (lag_1, lag_24) → keçmiş load dəyərlərini istifadə edərək zaman asılılığını modelə daxil edir
#Seasonality (month, season) → mövsümi dəyişiklikləri göstərir və uzunmüddətli patternləri tutur
#Region → coğrafi yerləşmə enerji istehlakının səviyyəsinə və patterninə birbaşa təsir edir, çünki hər regionda iqlim, sənaye fəaliyyəti və əhali sıxlığı fərqlidir